# RSNA Knee Abnormality Detection — EDA

Run after downloading CSVs: `scripts\download-csvs.bat`

In [ ]:
import sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from config import TARGET_LABELS
from src.data import load_train, load_train_series
from src.labels import labeled_mask, add_weak_labels

sns.set_theme(style="whitegrid")

In [ ]:
train = load_train()
series = load_train_series()
print(f"Studies: {len(train):,}")
print(f"Series: {len(series):,}")
print(f"Series per study (median): {series.groupby('StudyInstanceUID').size().median():.0f}")
train.head()

In [ ]:
mask = labeled_mask(train)
print(f"Labeled studies: {mask.sum():,} / {len(train):,} ({100*mask.mean():.1f}%)")

label_counts = train.loc[mask, TARGET_LABELS].notna().sum().sort_values(ascending=False)
label_counts.plot(kind="barh", figsize=(8, 5), title="Expert labels available per class")
plt.tight_layout()

In [ ]:
train["report_len"] = train["Report"].fillna("").str.len()
train["report_len"].hist(bins=50, figsize=(8, 4))
plt.title("Report length (characters)")
plt.xlabel("chars")

In [ ]:
series["Anatomical_Plane"].value_counts().plot(kind="bar", figsize=(6, 3), title="Series by plane")
plt.xticks(rotation=0)

In [ ]:
# Compare weak keyword labels vs expert labels on labeled subset
weak_df = add_weak_labels(train.loc[mask].head(500))
weak_cols = [f"weak_{c}" for c in TARGET_LABELS]
agree = (weak_df[weak_cols].values == weak_df[TARGET_LABELS].fillna(-1).values).mean()
print(f"Keyword agreement with expert labels (sample): {agree:.1%}")